# Modele j?zykowe - n-gramy 

---

## 1. N-gramy s??w w klasyfikacji
Poni?ej stworzono kod, kt?ry przeprowadza klasyfikacj? dokument?w nale??cych do 4 kategorii. W odr??nieniu do poprzednich zaj?? - tu zaproponowano klasyfikator SVC (algorytm SVM, popularna alternatywa dla NaiveBayes), kt?ra r?wnie? ?wietnie si? spisuje w problemach klasyfikacji tekstu.

**<span style="color: red">Zadanie 1a (0.5 punktu)</span>** Uruchom kod, przyjrzyj si? wygenerowanym wynikom (a najlepiej zachowaj je gdzie?, b?d? potrzebne). <br/>
Zapoznaj si? z dokumentacj? TfIdfVectorizer, odnajd? opcj? uwzgl?dnienia nie tylko pojedynczych s??w jako cechy, ale tak?e ich par i **zmodyfikuj poni?szy kod tak, aby klasyfikacja uwzgl?dnia?a zar?wno pojedyncze s?owa jak i pary (pozostaw parametr max_df=0.1 nienaruszony).** <span style="color: red"> Zmodyfikuj lini? 30.</span><br/> <br/>
**<span style="color: red">Zadanie 1b (0.5 punktu)</span>** Jak zmieni?a si? liczba cech po uwzgl?dnieniu tych par? Czy co? si? zmieni?o w raporcie z klasyfikacji? Uzupe?nij odpowiedzi na pytania w kom?rce poni?ej kodu.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.datasets import fetch_20newsgroups # zbi?r danych zawarty w Sklearn, kt?ry zawiera dane z 20 grup newsowych
import numpy as np

# ------------------- WCZYTANIE DANYCH -----------

np.random.seed(0) # ustaw seed na 0, aby zapewni? powtarzalno?? eksperymentu

categories = ['sci.space', 'comp.graphics', 'talk.politics.misc', 'comp.sys.mac.hardware'] # kategorie do analizy

train = fetch_20newsgroups(subset='train',
                                   categories=categories,
                                   shuffle=True,
                                   random_state=42) # pobieramy zbi?r ucz?cy (na nim b?dziemy trenowa?) dla wybranych kategorii.
    

test = fetch_20newsgroups(subset='test',
                                  categories=categories,
                                  shuffle=True,
                                  random_state=42) # pobieramy zbi?r testowy (na nim b?dziemy testowa?) dla wybranych kategorii



# ------------------- STWORZENIE PIPELINE'U -----------
    
pipeline = Pipeline([             # stw?rzmy pipeline surowy tekst -> TFIDF vectorizer -> klasyfikator 
    ('tfidf', TfidfVectorizer(max_df=0.1, ngram_range=(1, 2))),
    ('clf', SVC(C=1.0, kernel='linear')),
])

# ------------------- TRANSFORMACJA I UCZENIE -----------

pipeline.fit(train.data, train.target) # zwektoryzujmy dane i wytrenujmy klasyfikator na zbiorze treningowym


print("W s?owniku znajduje si? {n} r??nych cech".format(
    n=len(pipeline.named_steps['tfidf'].vocabulary_.keys())
))

# ------------------- OCENA KLASYFIKATORA -----------
print(classification_report(test.target, pipeline.predict(test.data))) # testowanie klasyfikatora - szerokie podsumowanie uwzgl?dniaj?ce miary: precision, recall, f1

## 2. N-gramy liter w klasyfikacji
Poza n-gramami stworzonymi z nast?puj?cych po sobie wyraz?w - bardzo cz?sto u?ywane s? r?wnie? n-gramy znakowe, stworzone z nast?puj?cych po sobie liter. <br/><br/>
Dla przyk?adu. wszystkie 3-gramy (trigramy) znakowe z napisu "Hello world" to: <br/>
"Hel", "ell", "llo", "lo ", "o w", " wo", "wor", "orl", "rld". <br/><br/>
Do czego mog?aby by? u?yta taka reprezentacja tekst?w? Okazuje si?, ?e ca?kiem mocno pomaga to w rozwi?zywaniiu problemu detekcji j?zyka w kt?rym zosta? zapisany dokument, szczeg?lnie w sytuacji, kiedy teksty s? bardzo kr?tkie (np. tweety, smsy).
<br/>
Poni?ej znajduje si? szkielet klasyfikatora rozpoznaj?cego j?zyk w kt?rym zapisany jest dokument.
J?zyk?w jest 6: polski, angielski, niemiecki, francuski, hiszpa?ski i w?oski.
<br/>
**<span style="color: red">Zadanie 2 (1 punkt)</span>**: Przedstawiony klasyfikator jest znanym ju? z poprzednich przyk?ad?w kodem. Waszym zadaniem jest:
<ol>
    <li>Zapoznanie si? dokumentacj? Tf-Idf vectorizera, aby znale?? funkcjonalno??, kt?ra zamiast ca?ych s??w, stworzy cechy na podstawie liter i wykorzystanie tej funkcjonalno?ci w kodzie</li>
    <li>Ustawienie takiego zakresu n-gram?w, aby zmaksymalizowa? uzyskany wynik (Oczekiwane 1.0 precyzji i recallu we wszystkich kategoriach przy pozostawieniu warto?i max_features = 300 element?w)</li>
    <li>Poprawnie zaklasyfikuje krotki przyk?ad zapisany w linii 43 (Bonjour przypisze do kategorii 'french').</li>
</ol>

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import pandas
import numpy as np

# -------------------- FUNKCJE POMOCNICZNE --------

# Funkcja mapuj?ca identyfikator liczbowy kategorii na warto?? tekstow?, np: 0->"polish", 1->"english"
def get_class_name_from_id(ids, mapping):
    return [mapping[id] for id in ids]
# ------------------- WCZYTANIE DANYCH -----------
full_dataset = pandas.read_csv('data/language_detection_1000.csv', encoding='utf-8') # wczytaj dane z pliku CSV
lang_to_id = {'polish': 0, 'english': 1, 'french': 2,
              'german': 3, 'italian': 4, 'spanish': 5}
id_to_lang = {v: k for k,v in lang_to_id.items()}
full_dataset['label_num'] = full_dataset.lang.map(lang_to_id)  # poniewa? nazwy kategorii zapisane s? z u?yciem string?w: "ham"/"spam", wykonujemy mapowanie tych warto?ci na liczby, aby m?c wykona? klasyfikacj?. 

np.random.seed(0)                                       # ustaw seed na 0, aby zapewni? powtarzalno?? eksperymentu
train_indices = np.random.rand(len(full_dataset)) < 0.7 # wylosuj 70% wierszy, kt?re znajd? si? w zbiorze treningowym

train = full_dataset[train_indices] # wybierz zbior treningowy (70%)
test = full_dataset[~train_indices] # wybierz zbi?r testowy (dope?nienie treningowego - 30%)


# ------------------- STWORZENIE PIPELINE'U -----------  
pipeline = Pipeline([             # stw?rzmy pipeline surowy tekst -> TFIDF vectorizer -> klasyfikator 
    ('tfidf', TfidfVectorizer(max_features=300, analyzer='char_wb', ngram_range=(2,3))),
    ('scaler', StandardScaler(with_mean = False)),
    ('clf', LogisticRegression()),
])
# ------------------- TRANSFORMACJA I UCZENIE -----------

pipeline.fit(train['text'], train['label_num']) # zwektoryzujmy dane i wytrenujmy klasyfikator na zbiorze treningowym

print("Oto kilka przyk?adowych cech stworzonych przez TfidfVectorizer: {n}".format(
    n=list(pipeline.named_steps['tfidf'].vocabulary_.keys())[:5]))

# ------------------- WERYFIKACJA NA KR?TKIM TEK?CIE ----

text_to_predict = "C?mo est?s?"
predicted = pipeline.predict([text_to_predict])
print("\n\nTekst: {t} zosta? zaklasyfikowany jako: {p}\n\n".format(
    t=text_to_predict,
    p=id_to_lang[predicted[0]]
))


# ------------------- OCENA KLASYFIKATORA -----------
print(classification_report(
    get_class_name_from_id(test['label_num'], id_to_lang), 
    get_class_name_from_id(pipeline.predict(test['text']), id_to_lang)
))

Widzimy, ?e problem jest stosunkowo prosty. Po co zatem u?ywa? n-gram?w znakowych? Aby zaoszcz?dzi? pami?? i podo?a? sytuacjom, w kt?rych zbi?r testowy sk?ada si? ze s??w, kt?re nie wyst?puj? w korpusie ucz?cym. <br/>

O ile wszystkich s??w w tych 6 j?zykach jest "30078", to trigram?w znakowych jest ju? tylko "15274", a bigram?w - "2059". W zwi?zku z tym: <ol>
<li>U?ywaj?c n-gram?w znakowych cz?sto mo?emy ograniczy? liczb? cech</li>
<li>N-gramy znakowe pomog? nam w sytuacjach, kiedy dane s?owo nie wyst?pi?o w tek?cie ucz?cym. Je?li opieramy uczenie na pe?nych s?owach i ca?y nasz tekst testowy to niewyst?puj?ce w korpusie ucz?cym - "bonjour", wtedy wektor BOW b?dzie zawiera? same zera, przez co b?dzie mia? problem z przydzia?em do odpowiedniej klasy. <br/> N-gramy znakowe nawet je?li nie napotka?y danego s?owa podczas analizy korpusu, to na podstawie budowy samego s?owa s? w stanie przewidywa? do jakiego j?zyka s?owo nale?y. Np. cokolwiek zawieraj?cego trigram "?eb" nale?e? b?dzie raczej do j?zyka polskiego.</li>
</ol>

---

### 2a - istotno?? cech

Poniewa? w zadaniu 2 u?yli?my znanego z zaj?? z przedmiotu "Sztuczna Inteligencja" klasyfikatora liniowego - regresji logistycznej, podejrze? mo?emy jakie cechy najsilniej sugeruj? nam przynale?no?? do danej klasy. Uruchom poni?szy kod, aby zobaczy? jakie cechy s? najwa?niejsze dla danych kategorii. Modyfikuj?c parametry TfidfVectorizer mo?esz zobaczy? jakie s?owa/ci?gi znak?w s? najistotniejsze do detekcji danego j?zyka.

In [ ]:
# Funkcja, z u?yciem kt?rej mo?emy oceni? kt?re cechy najsilniej skojarzone s? z danymi klasami.
# Wy?wietli list? s??w/n-gram?w znakowych, kt?rych obecno?? najsilniej wp?ywa na przydzia? do danej klasy
def language_indicators(feature_names, feature_importances, id_to_lang):
    for i, language in enumerate(feature_importances): # iterujemy po macierzy feature_importances (wymiar?w: j?zyk x cechy) wierszami (czyli j?zyk po j?zyku)
        scored_features = list(zip(feature_names, language)) # tworzymy skojarzenie nazw cech z wagami modelu (poniewa? u?ywamy regresji logistycznej - ka?da cecha (s?owo/n-gram) ma swoj? wag?, kt?ra jest optymalizowana w procesie uczenia! Cechy z wysokimi wagami s? wa?ne dla danej klasy. Ka?da klasa ma osobny model ze swoimi wagami!)
        scored_features = sorted(scored_features, key=lambda x: x[1], reverse=True) # posortujmy cechy skojarzone z wagami malej?co 
        print("W rozpoznaniu j?zyka {lang} najwa?niejsze cechy to:".format(
            lang=id_to_lang[i]) #zamie?my identyfikator numeryczny kategorii na nazw? j?zyka
        )
        for feature, score in scored_features[:5]: # wybierzmy po 5 najwa?niejszych cech (najwy?sze warto?ci uczonych wsp??czynnik?w)
            print("\t'{feature}': {score}".format(feature=feature, score=score))
        

# ------------------- WY?WIETLENIE NAJWA?NIEJSZYCH CECH DLA KA?DEJ KATEGORII
language_indicators(
    pipeline.named_steps['tfidf'].get_feature_names(), # pobierz nazwy cech
    pipeline.named_steps['clf'].coef_, # pobierz wyuczone wsp??czynniki (regresja logistyczna to stworzenie modelu opisanego wzorem y = e^(-wx - b), gdzie uczymy si? wsp??czynnik?w w. Pole coef_ zawiera te wsp??czynniki dla ka?dego j?zyka z osobna)
    id_to_lang # mapowanie z identyfikatora numerycznego na pe?n? nazw? j?zyka - zwi?ksza czytelno?? wygenerowanego raportu
)

---

## 3. N-gramy s??w w generowaniu tekstu

Innym, bardzo ciekawym zastosowaniem n-gram?w jest mo?liwo?? generowania tekstu z u?yciem tzw. ?a?cuch?w Markova. Stw?rzmy funkcj? generuj?c? n-gramy s??w, aby p??niej m?c j? wykorzysta? do tworzenia tekst?w.

**<span style="color: red">Zadanie 3 (1 punkt)</span>** stw?rz funkcj?, kt?ra wygeneruje n-gramy s??w zadanego stopnia n (n_gram_len). Aby podzieli? zdanie na s?owa nie musisz u?ywa? tokenizatora z biblioteki, na potrzeby zadania wystarczy uzna?, ?e spacja oddziela poszczeg?lne s?owa.
<br/>
<br/>
<div class="alert alert-success">
Oczekiwany rezultat dla zadanych danych: <br/><br/>[['The', 'big', 'brown'], ['big', 'brown', 'fox'], ['brown', 'fox', 'jumped'], ['fox', 'jumped', 'over'], ['jumped', 'over', 'the'], ['over', 'the', 'fence.']]
</div>
<br/>

In [ ]:
def get_word_ngrams(data, n_gram_len):
#     tfv = TfidfVectorizer(ngram_range=(n_gram_len, n_gram_len))
#     tfv.fit([data]) # zwektoryzujmy dane i wytrenujmy klasyfikator na zbiorze treningowym
#     return [x.split(" ") for x in tfv.vocabulary_.keys()]

    splitted = data.split(' ')
    res = []
    for i in range(len(splitted) - (n_gram_len - 1)):
        res.append([splitted[x] for x in range(i, i+n_gram_len)])
    return res

print(get_word_ngrams("The big brown fox jumped over the fence.", 3))

Je?li uda?o Ci si? napisa? funkcj? get_word_ngrams - zapoznaj si? z poni?szym kodem i uruchom go, aby wytworzy? tekst!

In [ ]:
from collections import Counter
import random
import itertools

def generate_ngram_markov(n_gram_len):
    markov_dict = dict() # stw?rz s?ownik, kt?ry wska?e list? dozwolonych s??w po zaobserwowanych n-poprzednich s?owach.
    with open("data/polish_europarl.txt", 'r') as f: # wczytaj korpus danych
        data = f.read().lower()                                           # zamie? wszystkie wielkie litery na ma?e
        n_grams = get_word_ngrams(data, n_gram_len)                   # wygeneruj wszystkie n-gramy s??w z korpusu
        for n_gram in n_grams:                   # dla ka?dego n-gramu...
            context = " ".join(n_gram[:-1])      # we? wszystkie s?owa z n-gramu poza ostatnim i po??cz w 1 string spacj?
            last_word = str(n_gram[-1])          # we? ostatnie s?owo n-gramu
            
            if context not in markov_dict.keys(): # je?li n-gram ubiedzony o ostatnie s?owo nie wyst?puje jeszcze w s?owniku
                markov_dict[context] = list()     # to dopiszmy go do s?ownika i stw?rzmy mu list?
            markov_dict[context].append(last_word) # wiedz?c, ?e ubiedzony n-gram jest w s?owniku - dopiszmy ostatnie s?owo do listy
    
    for context in markov_dict.keys():                        # dla ka?dego kontekstu (ubiedzonego n-gramu)
        markov_dict[context] = Counter(markov_dict[context])  # stw?rz histogram s??w jakie wyst?puj? w korpusie po tym kontek?cie
    
    return markov_dict


n_gram_len = 2  # liczba s??w do stworznia n-gramu
markov_dict = generate_ngram_markov(n_gram_len)  # stworzenie s?ownika z histogramami s??w dla poszczeg?lnych kontekst?w

text = 'Podpu?kownik wyst?pi? w parlamencie' # tekst, od kt?rego zaczniemy generowanie

for i in range(500):               # powt?rzmy 500 razy ...
    text_spl = text.split(" ")     # podzielmy istniej?cy tekst po spacji (przeprowad?my naiwn? 'tokenizacj?')
    context = " ".join(text_spl[-n_gram_len+1:])   # pobierzmy ostatnie n_gram_len - 1 s??w
    idx = random.randrange(sum(markov_dict[context].values())) # sprawd?my s?owa, kt?re s? dozwolone jako nast?pniki naszego kontekstu (context) i wybierzmy taki nast?pnik, kt?ry zostanie wylosowany zgodnie z rozk?adem stworzonym przez histogram.
    new_word = next(itertools.islice(markov_dict[context].elements(), idx, None)) # wybierzmy wylosowane s?owo
    text = text + " " + new_word # doklejmy wylosowane s?owo na ko?cu
print(text)

---

## 4. N-gramy znakowe w generowaniu tekstu

W bardzo podobny spos?b do zadania 3, mo?emy stworzy? model, kt?ry generowa? b?dzie tekst literka po literce. <br/>
**<span style="color: red">Zadanie 4 (1 punkt)</span>** stw?rz funkcj?, kt?ra wygeneruje n-gramy znakowe zadanego stopnia n (n_gram_len).
<br/>
<br/>
<div class="alert alert-success">
Oczekiwany rezultat dla zadanych danych: ['The', 'he ', 'e b', ' bi', 'big', 'ig ', 'g b', ' br', 'bro', 'row', 'own', 'wn ', 'n f', ' fo', 'fox', 'ox ', 'x j', ' ju', 'jum', 'ump', 'mpe', 'ped', 'ed ', 'd o', ' ov', 'ove', 'ver', 'er ', 'r t', ' th', 'the', 'he ', 'e f', ' fe', 'fen', 'enc', 'nce', 'ce.']
</div>
<br/>

In [ ]:
def get_character_ngrams(data, n_gram_len):
    res = []
    for i in range(len(data) - (n_gram_len - 1)):
        n_gram = ''
        for x in range(i, i+n_gram_len):
            n_gram += data[x]
        res.append(n_gram)
    return res
print(get_character_ngrams("The big brown fox jumped over the fence.", 3))

Po stworzeniu funkcji **get_character_ngrams()** mo?emy uruchomi? generator znakowy.

In [ ]:
from collections import Counter
import random
import itertools

def generate_ngram_markov(n_gram_len):
    markov_dict = dict() # stw?rz s?ownik, kt?ry wska?e list? dozwolonych s??w po zaobserwowanych n-poprzednich s?owach.
    with open('data/pan_tadeusz.txt', 'r') as f: # wczytaj korpus danych
        data = f.read().lower()                                           # zamie? wszystkie wielkie litery na ma?e
        n_grams = get_character_ngrams(data, n_gram_len)                   # wygeneruj wszystkie n-gramy s??w z korpusu
        for n_gram in n_grams:                   # dla ka?dego n-gramu...
            context = n_gram[:-1]  # we? wszystkie znaki n-gramu poza ostatnim 
            last_char = n_gram[-1] # we? ostatni znak n-gramu
            if context not in markov_dict.keys(): # je?li n-gram ubiedzony o ostatni znak nie wyst?puje jeszcze w s?owniku
                markov_dict[context] = list()     # to dopiszmy go do s?ownika i stw?rzmy mu list?
            markov_dict[context].append(last_char) # wiedz?c, ?e ubiedzony n-gram jest w s?owniku - dopiszmy ostatni znak do listy
    
    for context in markov_dict.keys():                        # dla ka?dego kontekstu (ubiedzonego n-gramu)
        markov_dict[context] = Counter(markov_dict[context])  # stw?rz histogram liter jakie wyst?puj? w korpusie po tym kontek?cie
    
    return markov_dict


text = "Bigos grzano" # tekst, od kt?rego zaczniemy generowanie
n_gram_len = len(text)  # liczba znak?w do stworznia n-gramu
markov_dict = generate_ngram_markov(n_gram_len)  # stworzenie s?ownika z histogramami s??w dla poszczeg?lnych kontekst?w

for i in range(500):               # powt?rzmy 500 razy ...
    context = text[-n_gram_len+1:]   # pobierzmy ostatnie n_gram_len - 1 s??w
    idx = random.randrange(sum(markov_dict[context].values())) # sprawd?my s?owa, kt?re s? dozwolone jako nast?pniki naszego kontekstu (context) i wybierzmy taki nast?pnik, kt?ry zostanie wylosowany zgodnie z rozk?adem stworzonym przez histogram.
    new_char = next(itertools.islice(markov_dict[context].elements(), idx, None)) # wybierzmy wylosowane s?owo
    text = text + new_char # doklejmy wylosowany znak na ko?cu

print(text)

---

## 5. Ngramy do generowania tekstu - d?ugo?? ngramu a jako?? tekstu
<span style="color: red">**Zadanie 5 (1 punkt)**</span>
Obsweruj?c wyniki z zada? 3 i 4 i sprawdzaj?c r??ne d?ugo?ci n-gram?w (znakowych i s??w) zastan?w si?:
<ol>
<li>Jakie ryzyko w kontek?cie jako?ci tekstu niesie ze sob? tworzenie tekstu z bardzo **kr?tkich** n-gram?w?</li>
<li>Jakie ryzyko w kontek?cie jako?ci tekstu niesie ze sob? tworzenie tekstu z bardzo **d?ugich** n-gram?w?</li>
</ol>
Odpowiedzi zawrzyj w komentarzu poni?ej

In [ ]:
# Zad 5:
#  Pytanie 1: Kr?tkie n-gramy powoduj? stworzenie tekstu, kt?ry ma mniej sensu, wygl?da jak zbitek losowych s??w. W przypadku zadania 4 przy zawarciu kilku kr?tkich s??w algorytm radzi? sobie lepiej ni? przy jednym d?ugim.
#  Pytanie 2: Model przepisuje bardzo d?ugie fragmenty oryginalnego tekstu. Istnieje r?wnie? ryzyko nieznalezienia s??w pocz?tkowych w s?owniku.

---

## 6. Bonus: Prawdopodobie?stwo wyst?pienia zdania - bez punkt?w
Dodatkowo ciekawym zastosowaniem n-gram?w jest r?wnie? ocena - jak bardzo prawdopodobnym jest wyst?pienie danego zdania w rzeczywisto?ci. Kiedy rozwi?zujemy zadanie translacji mowy na tekst, spotykamy si? z sytuacjami, w kt?rych nie do ko?ca wiemy, czy s?owo, kt?re zosta?o wypowiedziane to np. "morze" czy "mo?e". Model j?zykowy oparty o n-gramy mo?e oceni? szans? wyst?pienia danego ci?gu wyraz?w, a wi?c r?wnie? wybra? bardziej prawdopodobny ci?g wyraz?w w danym j?zyku. <br/>

Bior?c pod uwag?, ?e zdanie to ci?g wyraz?w    $w_1, w_2, w_3, ..., w_n$
Mo?emy poczyni? upraszczaj?ce za?o?enie, ?e aktualne s?owo zale?ne jest jedynie od s?owa poprzedniego, zatem prawdopodobie?stwo wyst?pienia zdania $P(sentence) = p(w_1|beginOfSentence)*p(w_2|w_1)*p(w_3|w_2)*...*p(w_n|w_(n-1))$

Obliczaj?c prawdopodobie?stwa warunkowe, mo?e si? okaza?, ?e w testowanym przez nas zdaniu mog? wyst?pi? dwie problematyczne sytuacje:
<ol>
    <li>s?owo konteksowe $w_c$ ze wzoru $p(w_n|w_c)$ nie wyst?puje w korpusie - bardzo ma?a szansa je?li korpus jest wystarczaj?co du?y</li>
    <li>s?owo nast?puj?ce po kontekstowym ($w_n$) nie wsp??wyst?puje w korpusie ze s?owem kontekstowym (wi?c $p(w_n|w_c) = 0$ - ca?kiem mo?liwy stan, do?? ?atwo mo?na sobie wyobrazi? sytuacj? braku wsp??wyst?powania pewnych s??w nawet przetwarzaj?c du?y korpus</li>
</ol>

Aby poradzi? sobie z sytuacj?, w kt?rej chcemy aby pewne s?owo rozpoczyna?o/ko?czy?o tekst, mo?emy doda? sztuczne znaczniki pocz?tku (BOS - Begin of Sentence) i ko?ca (EOS - End of Sentence) zdania. Wprowadzaj?c te znaczniki, b?dziemy mogli obliczy? prawdopodobie?stwo wyst?pienia s?owa, pod warunkiem, ?e rozpoczyna ono zdanie $p(w_n|BOS)$

Poni?ej znajduje si? kod oceniaj?cy prawdopodobie?stwo wyst?pienia zda?. Widzimy, ?e jedno z tych zda? ma sensowniejszy tekst i chcieliby?my, aby komputer by? w stanie wybra? sensowniejsz? opcj?.

Problematyczne sytuacje rozwi?zane zosta?y nast?puj?co:
<ol>
<li>Je?li brak s?owa kontekstowego w wygenerowanym modelu - uznaj, ?e prawdopodobie?stwo ca?ego zania wynosi 0</li>
<li>Je?li s?owo nast?puj?ce po kontekstowym nie wsp??wyst?puje z kontekstowym - u?yj wyg?adzania aby ustawi? prawdopodobie?stwo na bardzo ma?? warto?? (ale niezerow?)</li>
</ol>

**Zapoznaj si? z kodem i urochom go, tutaj nie trzeba nic zmienia?, to jedynie wizualizacja zastosowania. Uwaga - aby poprawnie oszacowa? prawdopodobie?stwa potrzeba posiada? wykonane zadanie 3 (stworzona funkcja get_word_ngrams)**


In [ ]:
text1 = "i heard that the european union is a valuable concept." # tekst do oceny
text2 = "i had that the euro bean union is a variable concept."  # tekst do oceny

from nltk import sent_tokenize                                   # b?dziemy dzieli? na zdania
import re                                                        # i czy?ci? tekst

from collections import Counter, defaultdict
import random
import itertools

markov_dict = defaultdict(list)               # s?ownik zawieraj?cy cz?sto?ci wyst?powania s??w w zale?no?ci od poprzedzaj?cego je s?owa

def clean_text(text):
    return re.sub("[\n\t ]+", " ", text) # czyszczenie tekstu ze znak?w nowej linii, tabulator?w, spacji (wielokrotnych)

def make_begin_end_of_sentences(text):
    result = ""
    for context in markov_dict.keys():                        # dla ka?dego kontekstu
        markov_dict[context] = Counter(markov_dict[context])  # stw?rz histogram s??w jakie wyst?puj? w korpusie po tym kontek?cie
    
    probability_of_sent1 = get_sentence_probability(text1, markov_dict) # wyznacz prawdopodobie?stwo wyst?pienia text1
    probability_of_sent2 = get_sentence_probability(text2, markov_dict) # wyznacz prawdopodobie?stwo wyst?pienia text2
    
    print("Prawdopodobie?stwo wyst?pienia zdania 1: {p}".format(p=get_sentence_probability(text1, markov_dict)))
    print("Prawdopodobie?stwo wyst?pienia zdania 2: {p}".format(p=get_sentence_probability(text2, markov_dict)))
